# 2.2 — TabNet baseline cho is_fraud

Notebook này huấn luyện TabNet như mô hình Deep Learning thứ hai, dùng cùng tập validation stratified 80/20 và seed 9 như MLP task 2.1. Để bảo đảm cùng đúng các dòng, hãy đính kèm output của notebook 2.1 làm input; notebook sẽ nạp split_ids_seed9.csv từ input đó.

Chỉ train.csv được dùng ở task này. Không đọc test labels, không dò leaderboard và không dùng LLM.

In [ ]:
# Cài đúng phiên bản thư viện nếu môi trường Kaggle chưa có.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pytorch_tabnet") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "pytorch-tabnet==4.1.0"
    ])

import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from pytorch_tabnet.tab_model import TabNetClassifier

SEED = 9
TARGET = "is_fraud"
ID_COL = "id"
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e11")
OUTPUT_DIR = Path("/kaggle/working/task_2_2_tabnet")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("TabNet package:", __import__("pytorch_tabnet").__version__ if hasattr(__import__("pytorch_tabnet"), "__version__") else "4.1.0")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2.2.1–2.2.3 — Nạp dữ liệu, cố định split và tiền xử lý

Notebook tìm split_ids_seed9.csv trong input đã gắn từ task 2.1 rồi ghép theo id. Vì vậy membership train/validation không phụ thuộc thứ tự dòng hoặc một lần chia ngẫu nhiên mới. Cột id và target bị loại khỏi feature; scaler và bảng mã category chỉ fit trên phần train.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
split_files = list(Path("/kaggle/input").rglob("split_ids_seed9.csv"))
if len(split_files) != 1:
    raise FileNotFoundError(
        "Cần gắn output Kaggle của task 2.1 làm input để tìm duy nhất "
        "split_ids_seed9.csv."
    )

split_ids = pd.read_csv(split_files[0])
if not {ID_COL, "split"}.issubset(split_ids.columns):
    raise ValueError("split_ids_seed9.csv phải có cột id và split.")
if split_ids[ID_COL].duplicated().any():
    raise ValueError("split_ids_seed9.csv có id bị trùng.")

train = train.merge(
    split_ids[[ID_COL, "split"]],
    on=ID_COL,
    how="left",
    validate="one_to_one",
    sort=False,
)
if train["split"].isna().any():
    raise ValueError("Có dòng train chưa được gán membership trong split seed 9.")
if set(train["split"].unique()) != {"train", "validation"}:
    raise ValueError("Giá trị split phải gồm train và validation.")

train_part = train.loc[train["split"].eq("train")].copy()
valid_part = train.loc[train["split"].eq("validation")].copy()
y_train = train_part[TARGET].astype(np.int64).to_numpy()
y_valid = valid_part[TARGET].astype(np.int64).to_numpy()

X_train_raw = train_part.drop(columns=[ID_COL, TARGET, "split"]).copy()
X_valid_raw = valid_part.drop(columns=[ID_COL, TARGET, "split"]).copy()
cat_cols = X_train_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [col for col in X_train_raw.columns if col not in cat_cols]

for frame in (X_train_raw, X_valid_raw):
    for col in cat_cols:
        frame[col] = frame[col].astype("string").fillna("__MISSING__")
    for col in num_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

numeric_medians = X_train_raw[num_cols].median()
X_train_raw[num_cols] = X_train_raw[num_cols].fillna(numeric_medians)
X_valid_raw[num_cols] = X_valid_raw[num_cols].fillna(numeric_medians)

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_raw[num_cols]).astype(np.float32)
X_valid_num = scaler.transform(X_valid_raw[num_cols]).astype(np.float32)

category_mappings = {}
X_train_cat = np.zeros((len(X_train_raw), len(cat_cols)), dtype=np.float32)
X_valid_cat = np.zeros((len(X_valid_raw), len(cat_cols)), dtype=np.float32)
cat_dims = []
for col_idx, col in enumerate(cat_cols):
    categories = sorted(X_train_raw[col].astype(str).unique().tolist())
    mapping = {value: index + 1 for index, value in enumerate(categories)}
    category_mappings[col] = mapping
    X_train_cat[:, col_idx] = X_train_raw[col].astype(str).map(mapping).to_numpy(dtype=np.float32)
    X_valid_cat[:, col_idx] = (
        X_valid_raw[col].astype(str).map(mapping).fillna(0).to_numpy(dtype=np.float32)
    )
    cat_dims.append(len(mapping) + 1)  # 0 dành cho category lạ ở validation.

X_train = np.concatenate([X_train_num, X_train_cat], axis=1)
X_valid = np.concatenate([X_valid_num, X_valid_cat], axis=1)
cat_idxs = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

assert X_train.shape[0] == len(y_train)
assert X_valid.shape[0] == len(y_valid)
assert np.isfinite(X_train).all() and np.isfinite(X_valid).all()

print("Split file:", split_files[0])
print("Rows train/validation:", len(train_part), len(valid_part))
print("Fraud rate train/validation:", round(float(y_train.mean()), 6), round(float(y_valid.mean()), 6))
print("Numeric/categorical features:", len(num_cols), len(cat_cols))
print("TabNet input shape:", X_train.shape)
print("Categorical indexes/dimensions:", cat_idxs, cat_dims)

## 2.2.4–2.2.5 — Huấn luyện TabNet và lưu dự đoán validation

Cấu hình này dùng TabNetClassifier với attention/decision dimensions 16, bốn decision steps, Adam và early stopping theo validation ROC-AUC. Không dùng class weighting ở baseline TabNet; việc đó được khảo sát riêng trong task 2.3. Checkpoint tốt nhất được lưu bởi thư viện.

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

model = TabNetClassifier(
    n_d=16,
    n_a=16,
    n_steps=4,
    gamma=1.3,
    lambda_sparse=1e-4,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={"lr": 2e-2},
    mask_type="entmax",
    seed=SEED,
    verbose=10,
)

start_time = time.time()
model.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=["valid"],
    eval_metric=["auc"],
    max_epochs=40,
    patience=5,
    batch_size=4096,
    virtual_batch_size=512,
    num_workers=0,
    drop_last=False,
)
training_seconds = time.time() - start_time

valid_probability = model.predict_proba(X_valid)[:, 1].astype(np.float32)
valid_roc_auc = float(roc_auc_score(y_valid, valid_probability))
valid_average_precision = float(average_precision_score(y_valid, valid_probability))

model.save_model(str(OUTPUT_DIR / "tabnet_seed9"))
joblib.dump(
    {
        "scaler": scaler,
        "numeric_medians": numeric_medians.to_dict(),
        "category_mappings": category_mappings,
        "numeric_columns": num_cols,
        "categorical_columns": cat_cols,
        "categorical_indexes": cat_idxs,
        "categorical_dimensions": cat_dims,
        "seed": SEED,
    },
    OUTPUT_DIR / "preprocessor.joblib",
)

history_data = {}
for key, values in model.history.history.items():
    history_data[key] = list(values)
pd.DataFrame(history_data).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

pd.DataFrame(
    {
        ID_COL: valid_part[ID_COL].to_numpy(),
        TARGET: y_valid,
        "fraud_probability": valid_probability,
    }
).to_csv(OUTPUT_DIR / "validation_predictions_tabnet_seed9.csv", index=False)

metrics = {
    "task": "2.2 TabNet baseline",
    "seed": SEED,
    "split": "shared stratified 80/20 split from task 2.1",
    "n_train": int(len(y_train)),
    "n_validation": int(len(y_valid)),
    "positive_rate_train": float(y_train.mean()),
    "positive_rate_validation": float(y_valid.mean()),
    "numeric_features": num_cols,
    "categorical_features": cat_cols,
    "architecture": {"n_d": 16, "n_a": 16, "n_steps": 4, "gamma": 1.3, "lambda_sparse": 1e-4},
    "categorical_dimensions": cat_dims,
    "optimizer": "Adam",
    "learning_rate": 0.02,
    "batch_size": 4096,
    "virtual_batch_size": 512,
    "max_epochs": 40,
    "patience": 5,
    "best_epoch": int(getattr(model, "best_epoch", -1)),
    "epochs_run": int(len(model.history.history.get("loss", []))),
    "training_seconds": float(training_seconds),
    "validation_roc_auc": valid_roc_auc,
    "validation_average_precision": valid_average_precision,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
}
with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print("Best epoch:", metrics["best_epoch"])
print("Validation ROC-AUC:", round(valid_roc_auc, 6))
print("Validation average precision:", round(valid_average_precision, 6))
print("Training seconds:", round(training_seconds, 1))
print("Artifacts:", OUTPUT_DIR)

## 2.2.6–2.2.7 — Kết quả cần đối chiếu

Hai metric được tính trên cùng các dòng validation với MLP: ROC-AUC là metric chính, Average Precision/PR-AUC là metric phụ. File dự đoán giữ id, nhãn validation và xác suất để task 2.4 có thể so sánh theo từng giao dịch. Không dùng threshold để chấm Kaggle.